# Frequency Sampling Modules Testing

Test each frequency-sampling filter module independently against the traditional time-domain sample-based implementation:
1. **Dynamics Filter (1st order IIR)**
2. **Pluck Position Filter (All-zero Comb)**
3. **Karplus-Strong (With a 1st order IIR loop filter)**

In [1]:
import numpy as np
import numpy.typing as npt
from dataclasses import dataclass
from typing import Optional
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from scipy import signal
import IPython.display as ipd
import traceback

from synths import (
    pluck_position_filter,
    dynamics_filter,
    karplus_strong,
    Implementation,
    F0_MIN,
    FS_MIN,
    RND_SEED,
    LAGRANGE_ORDER,
    IIR_TRUNCATION,
    N_FFT,
    lin_resample
)

/Users/pablotablasdepaula/PycharmProjects/DAFx26-Karplus/.pixi/envs/default/lib/python3.13/site-packages/philtorch/__init__.py:11: UserWarning: Custom extension not loaded.
  warnings.warn("Custom extension not loaded.")
/Users/pablotablasdepaula/PycharmProjects/DAFx26-Karplus/.pixi/envs/default/lib/python3.13/site-packages/torchlpc/__init__.py:23: UserWarning: Custom extension not loaded. Falling back to Numba implementation.
  warnings.warn("Custom extension not loaded. Falling back to Numba implementation.")


In [2]:
@dataclass
class ParameterRange:
    """Defines valid range for a parameter."""
    min: float
    max: float

    def clip(self, value: npt.NDArray) -> npt.NDArray:
        return np.clip(value, self.min, self.max)

# Parameter ranges
PARAM_RANGES = {
    'f0': ParameterRange(F0_MIN, FS_MIN / 2.0),
    'position': ParameterRange(0.0, 1.0),
    'dynamic_level': ParameterRange(0.0, 1.0),
    'a1': ParameterRange(0.0, 1.0),
    'decay': ParameterRange(0.0, 1.0),
}

@dataclass
class LFOConfig:
    """Low Frequency Oscillator configuration."""
    rate: float  # Hz
    amplitude: float  # 0-1, fraction of parameter range

    def generate(self, num_frames: int, center: float, param_range: ParameterRange, duration: float) -> npt.NDArray:
        """Generate LFO-modulated parameter trajectory."""
        t = np.linspace(0, duration, num_frames)
        lfo = np.sin(2 * np.pi * self.rate * t)
        range_span = param_range.max - param_range.min
        modulation_depth = self.amplitude * range_span / 2.0
        return param_range.clip(center + lfo * modulation_depth)

def generate_continuous_noise(num_samples: int, seed: int = RND_SEED) -> torch.Tensor:
    """Generate continuous white noise for entire duration."""
    generator = torch.Generator()
    generator.manual_seed(seed)
    noise = torch.rand(1, num_samples, generator=generator)
    noise = (noise - 0.5) * 2.0  # [-1, 1]
    return noise

def generate_burst_noise(num_samples: int, num_frames: int, trigger_rate: float, duration: float, seed: int = RND_SEED) -> torch.Tensor:
    """Generate noise bursts at regular intervals based on trigger rate."""
    generator = torch.Generator()
    generator.manual_seed(seed)
    
    # Generate white noise for entire duration
    noise = torch.rand(1, num_samples, generator=generator)
    noise = (noise - 0.5) * 2.0  # [-1, 1]
    
    # Create gating envelope based on trigger rate
    num_triggers = int(duration * trigger_rate)
    trigger_interval_frames = num_frames / (duration * trigger_rate)
    samples_per_frame = num_samples / num_frames
    
    # Create envelope: 1.0 for burst_duration samples at each trigger, 0.0 elsewhere
    envelope = torch.zeros(1, num_samples)
    burst_samples = int(0.01 * FS_MIN)  # 10ms burst duration
    
    for i in range(num_triggers):
        frame_idx = int(i * trigger_interval_frames)
        sample_idx = int(frame_idx * samples_per_frame)
        if sample_idx < num_samples:
            end_idx = min(sample_idx + burst_samples, num_samples)
            envelope[0, sample_idx:end_idx] = 1.0
    
    return noise * envelope

def generate_parameter(param_name: str, center: float, lfo: Optional[LFOConfig], 
                      num_frames: int, duration: float) -> npt.NDArray:
    """Generate parameter trajectory (constant or LFO-modulated)."""
    param_range = PARAM_RANGES[param_name]
    if lfo is None:
        return np.full(num_frames, param_range.clip(np.array([center]))[0])
    return lfo.generate(num_frames, center, param_range, duration)

def compute_stats(ref: npt.NDArray, test: npt.NDArray, name: str) -> dict:
    """Compute comparison statistics."""
    diff = ref - test
    rms_ref = np.sqrt(np.mean(ref**2))
    rms_diff = np.sqrt(np.mean(diff**2))
    snr = 20 * np.log10(rms_ref / (rms_diff + 1e-10))
    max_abs_diff = np.max(np.abs(diff))
    correlation = np.corrcoef(ref, test)[0, 1]
    return {
        'name': name,
        'snr': snr,
        'rms_error': rms_diff,
        'max_diff': max_abs_diff,
        'correlation': correlation
    }

def plot_comparison(audio_td: npt.NDArray, audio_freq: npt.NDArray, 
                   params: dict, duration: float, fs: int, num_frames: int,
                   title: str, param_labels: list):
    """Create comparison plots."""
    from matplotlib.gridspec import GridSpec
    
    fig = plt.figure(figsize=(18, 12))
    gs = GridSpec(4, 3, figure=fig, hspace=0.35, wspace=0.3)
    
    time_signal = np.linspace(0, duration, len(audio_td))
    time_frames = np.linspace(0, duration, num_frames)
    
    # Waveforms
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(time_signal, audio_td, 'b-', lw=0.5, alpha=0.7, label='Time Domain')
    ax1.plot(time_signal, audio_freq, 'r-', lw=0.5, alpha=0.7, label='Frequency Domain')
    ax1.set_title('Waveform Comparison', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Amplitude')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Spectrograms
    for idx, (audio, label, pos) in enumerate([
        (audio_td, 'Time Domain', gs[1, 0]),
        (audio_freq, 'Frequency Domain', gs[1, 1]),
        (audio_td - audio_freq, 'Difference', gs[1, 2])
    ]):
        ax = fig.add_subplot(pos)
        f, t, Sxx = signal.spectrogram(audio, fs, nperseg=1024, noverlap=768)
        Sxx_db = 10 * np.log10(Sxx + 1e-10)
        img = ax.pcolormesh(t, f, Sxx_db, shading='gouraud', cmap='viridis')
        ax.set_title(label, fontsize=10, fontweight='bold')
        ax.set_ylim([0, 4000])
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Frequency (Hz)')
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
    
    # Error comparison
    ax_err = fig.add_subplot(gs[2, :])
    diff = audio_td - audio_freq
    ax_err.plot(time_signal, diff, 'g-', lw=0.5, alpha=0.7)
    ax_err.set_title('Waveform Difference (Time - Frequency)', fontsize=12, fontweight='bold')
    ax_err.set_xlabel('Time (s)')
    ax_err.set_ylabel('Amplitude')
    ax_err.grid(True, alpha=0.3)
    
    # Parameter trajectories
    num_params = len(param_labels)
    for idx, (param_name, label, is_modulated) in enumerate(param_labels):
        ax = fig.add_subplot(gs[3, idx])
        values = params[param_name]
        color = 'red' if is_modulated else 'blue'
        ax.plot(time_frames, values, color=color, lw=2, alpha=0.8)
        ax.set_title(label, fontsize=10, fontweight='bold')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3)
        param_range = PARAM_RANGES[param_name]
        ax.set_ylim([param_range.min, param_range.max])
    
    fig.suptitle(title, fontsize=14, fontweight='bold', y=0.995)
    plt.show()

## 1. Dynamics Filter - Continuous Noise

In [3]:
# Interactive controls for Dynamics Filter (Continuous Noise)
dyn_duration = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description='Duration (s)')
dyn_f0_center = widgets.FloatSlider(value=200.0, min=F0_MIN, max=500.0, step=10.0, description='f0 Center (Hz)')
dyn_dl_center = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Dynamic Level')
dyn_lfo_rate = widgets.FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='LFO Rate (Hz)')
dyn_lfo_amp = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description='LFO Amplitude')
dyn_f0_modulated = widgets.Checkbox(value=False, description='Route LFO to f0')
dyn_dl_modulated = widgets.Checkbox(value=False, description='Route LFO to dynamic level')

dyn_generate_button = widgets.Button(description='🎵 Generate & Compare', button_style='success')
dyn_output = widgets.Output()

def generate_dyn_comparison(b):
    with dyn_output:
        clear_output(wait=True)
        try:
            duration = dyn_duration.value
            fs = FS_MIN
            num_frames = 250
            num_samples = int(duration * fs)
            
            lfo = LFOConfig(rate=dyn_lfo_rate.value, amplitude=dyn_lfo_amp.value) if (dyn_f0_modulated.value or dyn_dl_modulated.value) else None
            
            f0_np = generate_parameter('f0', dyn_f0_center.value, lfo if dyn_f0_modulated.value else None, num_frames, duration)
            dl_np = generate_parameter('dynamic_level', dyn_dl_center.value, lfo if dyn_dl_modulated.value else None, num_frames, duration)
            
            params = {'f0': f0_np, 'dynamic_level': dl_np}
            
            f0_torch = torch.from_numpy(f0_np).float().unsqueeze(0)
            dl_torch = torch.from_numpy(dl_np).float().unsqueeze(0)
            
            f0_upsampled = lin_resample(f0_torch, num_samples)
            dl_upsampled = lin_resample(dl_torch, num_samples)
            
            # Generate continuous noise
            x = generate_continuous_noise(num_samples)
            
            print("⏳ Computing time-domain implementation...")
            audio_td = dynamics_filter(
                x=x,
                f0=f0_upsampled,
                dynamic_level=dl_upsampled,
                implementation=Implementation.DIFFABLE_TIME_DOMAIN,
                fs=fs
            ).squeeze(0).numpy()
            
            print("⏳ Computing frequency-domain implementation...")
            hop_length = N_FFT // 4
            window = torch.hann_window(N_FFT, device=x.device)
            
            X = torch.stft(x, n_fft=N_FFT, hop_length=hop_length, window=window, return_complex=True)
            num_stft_frames = X.shape[-1]
            X = X.permute(0, 2, 1)
            
            f0_stft = lin_resample(f0_torch, num_stft_frames)
            dl_stft = lin_resample(dl_torch, num_stft_frames)
            
            X_filtered = dynamics_filter(
                x=X,
                f0=f0_stft,
                dynamic_level=dl_stft,
                implementation=Implementation.FREQUENCY_SAMPLING,
                fs=fs,
                n_fft=N_FFT
            )
            
            X_filtered = X_filtered.permute(0, 2, 1)
            audio_freq = torch.istft(X_filtered, n_fft=N_FFT, hop_length=hop_length, 
                                    window=window, length=num_samples).squeeze(0).numpy()
            
            param_labels = [
                ('f0', 'f0 (Hz)', dyn_f0_modulated.value),
                ('dynamic_level', 'Dynamic Level', dyn_dl_modulated.value)
            ]
            
            modulated_params = [name for name, _, mod in param_labels if mod]
            mod_str = ', '.join(modulated_params) if modulated_params else 'None'
            title = f"Dynamics Filter Comparison (CONTINUOUS NOISE)\nLFO: {dyn_lfo_rate.value:.1f}Hz @ {dyn_lfo_amp.value:.0%} → [{mod_str}]"
            
            plot_comparison(audio_td, audio_freq, params, duration, fs, num_frames, title, param_labels)
            
            stats = compute_stats(audio_td, audio_freq, 'Time-Domain vs Frequency-Domain')
            print("\n" + "="*80)
            print("📊 COMPARISON STATISTICS")
            print("="*80)
            print(f"SNR:              {stats['snr']:>10.2f} dB")
            print(f"RMS Error:        {stats['rms_error']:>10.6f}")
            print(f"Max Abs Diff:     {stats['max_diff']:>10.6f}")
            print(f"Correlation:      {stats['correlation']:>10.8f}")
            match = '✅ MATCH' if stats['snr'] > 60 else '⚠️ DIFFERS'
            print(f"Match:            {match}")
            print("="*80)
            
            print("\n🔊 Audio Playback:")
            print("\nTime-Domain Implementation:")
            display(ipd.Audio(audio_td, rate=fs))
            print("\nFrequency-Domain Implementation:")
            display(ipd.Audio(audio_freq, rate=fs))
            print("\nDifference (Time - Freq):")
            display(ipd.Audio(audio_td - audio_freq, rate=fs))
            
        except Exception as e:
            clear_output(wait=False)
            print("\n" + "="*60)
            print("❌ ERROR OCCURRED:")
            print("="*60)
            print(f"Error type: {type(e).__name__}")
            print(f"Error message: {str(e)}")
            print("\nFull traceback:")
            traceback.print_exc()
            print("="*60)

dyn_generate_button.on_click(generate_dyn_comparison)

lfo_box = widgets.VBox([
    widgets.HTML("<h4>🌊 LFO Settings</h4>"),
    dyn_lfo_rate,
    dyn_lfo_amp,
])

params_box = widgets.VBox([
    widgets.HTML("<h4>🎛️ Parameters</h4>"),
    dyn_duration,
    widgets.HBox([dyn_f0_center, dyn_f0_modulated]),
    widgets.HBox([dyn_dl_center, dyn_dl_modulated]),
])

controls = widgets.VBox([
    widgets.HTML("<h3>🎹 Dynamics Filter Test</h3>"),
    widgets.HBox([lfo_box, params_box]),
    widgets.HTML("<br>"),
    dyn_generate_button,
], layout=widgets.Layout(padding='10px', border='2px solid #2196F3'))

display(controls, dyn_output)

Output()

## 2. Pluck Position Filter (Comb) - Continuous Noise

In [7]:
# Interactive controls for Pluck Position Filter (Continuous Noise)
pluck_duration = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description='Duration (s)')
pluck_f0_center = widgets.FloatSlider(value=200.0, min=F0_MIN, max=500.0, step=10.0, description='f0 Center (Hz)')
pluck_pos_center = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Position')
pluck_lfo_rate = widgets.FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='LFO Rate (Hz)')
pluck_lfo_amp = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description='LFO Amplitude')
pluck_f0_modulated = widgets.Checkbox(value=False, description='Route LFO to f0')
pluck_pos_modulated = widgets.Checkbox(value=False, description='Route LFO to position')

pluck_generate_button = widgets.Button(description='🎵 Generate & Compare', button_style='success')
pluck_output = widgets.Output()

def generate_pluck_comparison(b):
    with pluck_output:
        clear_output(wait=True)
        try:
            duration = pluck_duration.value
            fs = FS_MIN
            num_frames = 250
            num_samples = int(duration * fs)
            
            lfo = LFOConfig(rate=pluck_lfo_rate.value, amplitude=pluck_lfo_amp.value) if (pluck_f0_modulated.value or pluck_pos_modulated.value) else None
            
            f0_np = generate_parameter('f0', pluck_f0_center.value, lfo if pluck_f0_modulated.value else None, num_frames, duration)
            pos_np = generate_parameter('position', pluck_pos_center.value, lfo if pluck_pos_modulated.value else None, num_frames, duration)
            
            params = {'f0': f0_np, 'position': pos_np}
            
            f0_torch = torch.from_numpy(f0_np).float().unsqueeze(0)
            pos_torch = torch.from_numpy(pos_np).float().unsqueeze(0)
            
            f0_upsampled = lin_resample(f0_torch, num_samples)
            pos_upsampled = lin_resample(pos_torch, num_samples)
            
            # Generate continuous noise
            x = generate_continuous_noise(num_samples)
            
            print("⏳ Computing time-domain implementation...")
            audio_td = pluck_position_filter(
                x=x,
                f0=f0_upsampled,
                position=pos_upsampled,
                implementation=Implementation.DIFFABLE_TIME_DOMAIN,
                fs=fs,
                lagrange_order=LAGRANGE_ORDER
            ).squeeze(0).numpy()
            
            print("⏳ Computing frequency-domain implementation...")
            hop_length = N_FFT // 4
            window = torch.hann_window(N_FFT, device=x.device)
            
            X = torch.stft(x, n_fft=N_FFT, hop_length=hop_length, window=window, return_complex=True)
            num_stft_frames = X.shape[-1]
            X = X.permute(0, 2, 1)
            
            f0_stft = lin_resample(f0_torch, num_stft_frames)
            pos_stft = lin_resample(pos_torch, num_stft_frames)
            
            X_filtered = pluck_position_filter(
                x=X,
                f0=f0_stft,
                position=pos_stft,
                implementation=Implementation.FREQUENCY_SAMPLING,
                fs=fs,
                n_fft=N_FFT
            )
            
            X_filtered = X_filtered.permute(0, 2, 1)
            audio_freq = torch.istft(X_filtered, n_fft=N_FFT, hop_length=hop_length, 
                                    window=window, length=num_samples).squeeze(0).numpy()
            
            param_labels = [
                ('f0', 'f0 (Hz)', pluck_f0_modulated.value),
                ('position', 'Pluck Position', pluck_pos_modulated.value)
            ]
            
            modulated_params = [name for name, _, mod in param_labels if mod]
            mod_str = ', '.join(modulated_params) if modulated_params else 'None'
            title = f"Pluck Position Filter Comparison (CONTINUOUS NOISE)\nLFO: {pluck_lfo_rate.value:.1f}Hz @ {pluck_lfo_amp.value:.0%} → [{mod_str}]"
            
            plot_comparison(audio_td, audio_freq, params, duration, fs, num_frames, title, param_labels)
            
            stats = compute_stats(audio_td, audio_freq, 'Time-Domain vs Frequency-Domain')
            print("\n" + "="*80)
            print("📊 COMPARISON STATISTICS")
            print("="*80)
            print(f"SNR:              {stats['snr']:>10.2f} dB")
            print(f"RMS Error:        {stats['rms_error']:>10.6f}")
            print(f"Max Abs Diff:     {stats['max_diff']:>10.6f}")
            print(f"Correlation:      {stats['correlation']:>10.8f}")
            match = '✅ MATCH' if stats['snr'] > 60 else '⚠️ DIFFERS'
            print(f"Match:            {match}")
            print("="*80)
            
            print("\n🔊 Audio Playback:")
            print("\nTime-Domain Implementation:")
            display(ipd.Audio(audio_td, rate=fs))
            print("\nFrequency-Domain Implementation:")
            display(ipd.Audio(audio_freq, rate=fs))
            print("\nDifference (Time - Freq):")
            display(ipd.Audio(audio_td - audio_freq, rate=fs))
            
        except Exception as e:
            clear_output(wait=False)
            print("\n" + "="*60)
            print("❌ ERROR OCCURRED:")
            print("="*60)
            print(f"Error type: {type(e).__name__}")
            print(f"Error message: {str(e)}")
            print("\nFull traceback:")
            traceback.print_exc()
            print("="*60)

pluck_generate_button.on_click(generate_pluck_comparison)

lfo_box = widgets.VBox([
    widgets.HTML("<h4>🌊 LFO Settings</h4>"),
    pluck_lfo_rate,
    pluck_lfo_amp,
])

params_box = widgets.VBox([
    widgets.HTML("<h4>🎛️ Parameters</h4>"),
    pluck_duration,
    widgets.HBox([pluck_f0_center, pluck_f0_modulated]),
    widgets.HBox([pluck_pos_center, pluck_pos_modulated]),
])

controls = widgets.VBox([
    widgets.HTML("<h3>🎹 Pluck Position Filter Test</h3>"),
    widgets.HBox([lfo_box, params_box]),
    widgets.HTML("<br>"),
    pluck_generate_button,
], layout=widgets.Layout(padding='10px', border='2px solid #4CAF50'))

display(controls, pluck_output)

Output()

## 3. Karplus-Strong - Continuous Noise

In [8]:
# Interactive controls for Karplus-Strong (Continuous Noise)
ks_cont_duration = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description='Duration (s)')
ks_cont_f0_center = widgets.FloatSlider(value=200.0, min=F0_MIN, max=500.0, step=10.0, description='f0 Center (Hz)')
ks_cont_a1_center = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='a1 (Brightness)')
ks_cont_decay_center = widgets.FloatSlider(value=0.99, min=0.0, max=1.0, step=0.01, description='Decay (Sustain)')
ks_cont_lfo_rate = widgets.FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='LFO Rate (Hz)')
ks_cont_lfo_amp = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description='LFO Amplitude')
ks_cont_f0_modulated = widgets.Checkbox(value=False, description='Route LFO to f0')
ks_cont_a1_modulated = widgets.Checkbox(value=False, description='Route LFO to a1')
ks_cont_decay_modulated = widgets.Checkbox(value=False, description='Route LFO to decay')

ks_cont_generate_button = widgets.Button(description='🎵 Generate & Compare', button_style='success')
ks_cont_output = widgets.Output()

def generate_ks_cont_comparison(b):
    with ks_cont_output:
        clear_output(wait=True)
        try:
            duration = ks_cont_duration.value
            fs = FS_MIN
            num_frames = 250
            num_samples = int(duration * fs)
            
            lfo = LFOConfig(rate=ks_cont_lfo_rate.value, amplitude=ks_cont_lfo_amp.value) if (ks_cont_f0_modulated.value or ks_cont_a1_modulated.value or ks_cont_decay_modulated.value) else None
            
            f0_np = generate_parameter('f0', ks_cont_f0_center.value, lfo if ks_cont_f0_modulated.value else None, num_frames, duration)
            a1_np = generate_parameter('a1', ks_cont_a1_center.value, lfo if ks_cont_a1_modulated.value else None, num_frames, duration)
            decay_np = generate_parameter('decay', ks_cont_decay_center.value, lfo if ks_cont_decay_modulated.value else None, num_frames, duration)
            
            params = {'f0': f0_np, 'a1': a1_np, 'decay': decay_np}
            
            f0_torch = torch.from_numpy(f0_np).float().unsqueeze(0)
            a1_torch = torch.from_numpy(a1_np).float().unsqueeze(0)
            decay_torch = torch.from_numpy(decay_np).float().unsqueeze(0)
            
            f0_upsampled = lin_resample(f0_torch, num_samples)
            a1_upsampled = lin_resample(a1_torch, num_samples)
            decay_upsampled = lin_resample(decay_torch, num_samples)
            
            # Generate continuous noise
            x = generate_continuous_noise(num_samples)
            
            print("⏳ Computing time-domain implementation...")
            audio_td = karplus_strong(
                x=x,
                f0=f0_upsampled,
                a1=a1_upsampled,
                g=decay_upsampled,
                implementation=Implementation.DIFFABLE_TIME_DOMAIN,
                fs=fs,
                lagrange_order=LAGRANGE_ORDER,
                iir_truncation=IIR_TRUNCATION
            ).squeeze(0).numpy()
            
            print("⏳ Computing frequency-domain implementation...")
            hop_length = N_FFT // 4
            window = torch.hann_window(N_FFT, device=x.device)
            
            X = torch.stft(x, n_fft=N_FFT, hop_length=hop_length, window=window, return_complex=True)
            num_stft_frames = X.shape[-1]
            X = X.permute(0, 2, 1)
            
            f0_stft = lin_resample(f0_torch, num_stft_frames)
            a1_stft = lin_resample(a1_torch, num_stft_frames)
            decay_stft = lin_resample(decay_torch, num_stft_frames)
            
            X_filtered = karplus_strong(
                x=X,
                f0=f0_stft,
                a1=a1_stft,
                g=decay_stft,
                implementation=Implementation.FREQUENCY_SAMPLING,
                fs=fs,
                n_fft=N_FFT
            )
            
            X_filtered = X_filtered.permute(0, 2, 1)
            audio_freq = torch.istft(X_filtered, n_fft=N_FFT, hop_length=hop_length, 
                                    window=window, length=num_samples).squeeze(0).numpy()
            
            param_labels = [
                ('f0', 'f0 (Hz)', ks_cont_f0_modulated.value),
                ('a1', 'a1 (Brightness)', ks_cont_a1_modulated.value),
                ('decay', 'Decay (Sustain)', ks_cont_decay_modulated.value)
            ]
            
            modulated_params = [name for name, _, mod in param_labels if mod]
            mod_str = ', '.join(modulated_params) if modulated_params else 'None'
            title = f"Karplus-Strong Comparison (CONTINUOUS NOISE)\nLFO: {ks_cont_lfo_rate.value:.1f}Hz @ {ks_cont_lfo_amp.value:.0%} → [{mod_str}]"
            
            plot_comparison(audio_td, audio_freq, params, duration, fs, num_frames, title, param_labels)
            
            stats = compute_stats(audio_td, audio_freq, 'Time-Domain vs Frequency-Domain')
            print("\n" + "="*80)
            print("📊 COMPARISON STATISTICS")
            print("="*80)
            print(f"SNR:              {stats['snr']:>10.2f} dB")
            print(f"RMS Error:        {stats['rms_error']:>10.6f}")
            print(f"Max Abs Diff:     {stats['max_diff']:>10.6f}")
            print(f"Correlation:      {stats['correlation']:>10.8f}")
            match = '✅ MATCH' if stats['snr'] > 60 else '⚠️ DIFFERS'
            print(f"Match:            {match}")
            print("="*80)
            
            print("\n🔊 Audio Playback:")
            print("\nTime-Domain Implementation:")
            display(ipd.Audio(audio_td, rate=fs))
            print("\nFrequency-Domain Implementation:")
            display(ipd.Audio(audio_freq, rate=fs))
            print("\nDifference (Time - Freq):")
            display(ipd.Audio(audio_td - audio_freq, rate=fs))
            
        except Exception as e:
            clear_output(wait=False)
            print("\n" + "="*60)
            print("❌ ERROR OCCURRED:")
            print("="*60)
            print(f"Error type: {type(e).__name__}")
            print(f"Error message: {str(e)}")
            print("\nFull traceback:")
            traceback.print_exc()
            print("="*60)

ks_cont_generate_button.on_click(generate_ks_cont_comparison)

lfo_box = widgets.VBox([
    widgets.HTML("<h4>🌊 LFO Settings</h4>"),
    ks_cont_lfo_rate,
    ks_cont_lfo_amp,
])

params_box = widgets.VBox([
    widgets.HTML("<h4>🎛️ Parameters</h4>"),
    ks_cont_duration,
    widgets.HBox([ks_cont_f0_center, ks_cont_f0_modulated]),
    widgets.HBox([ks_cont_a1_center, ks_cont_a1_modulated]),
    widgets.HBox([ks_cont_decay_center, ks_cont_decay_modulated]),
])

controls = widgets.VBox([
    widgets.HTML("<h3>🎹 Karplus-Strong Test (Continuous Noise)</h3>"),
    widgets.HBox([lfo_box, params_box]),
    widgets.HTML("<br>"),
    ks_cont_generate_button,
], layout=widgets.Layout(padding='10px', border='2px solid #FF9800'))

display(controls, ks_cont_output)

Output()

## 4. Karplus-Strong - Burst Rate Control

In [9]:
# Interactive controls for Karplus-Strong (Burst Rate Control)
ks_burst_duration = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description='Duration (s)')
ks_burst_trigger_rate = widgets.FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='Burst Rate (Hz)')
ks_burst_f0_center = widgets.FloatSlider(value=200.0, min=F0_MIN, max=500.0, step=10.0, description='f0 Center (Hz)')
ks_burst_a1_center = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='a1 (Brightness)')
ks_burst_decay_center = widgets.FloatSlider(value=0.99, min=0.0, max=1.0, step=0.01, description='Decay (Sustain)')
ks_burst_lfo_rate = widgets.FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='LFO Rate (Hz)')
ks_burst_lfo_amp = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description='LFO Amplitude')
ks_burst_f0_modulated = widgets.Checkbox(value=False, description='Route LFO to f0')
ks_burst_a1_modulated = widgets.Checkbox(value=False, description='Route LFO to a1')
ks_burst_decay_modulated = widgets.Checkbox(value=False, description='Route LFO to decay')

ks_burst_generate_button = widgets.Button(description='🎵 Generate & Compare', button_style='success')
ks_burst_output = widgets.Output()

def generate_ks_burst_comparison(b):
    with ks_burst_output:
        clear_output(wait=True)
        try:
            duration = ks_burst_duration.value
            trigger_rate = ks_burst_trigger_rate.value
            fs = FS_MIN
            num_frames = 250
            num_samples = int(duration * fs)
            
            lfo = LFOConfig(rate=ks_burst_lfo_rate.value, amplitude=ks_burst_lfo_amp.value) if (ks_burst_f0_modulated.value or ks_burst_a1_modulated.value or ks_burst_decay_modulated.value) else None
            
            f0_np = generate_parameter('f0', ks_burst_f0_center.value, lfo if ks_burst_f0_modulated.value else None, num_frames, duration)
            a1_np = generate_parameter('a1', ks_burst_a1_center.value, lfo if ks_burst_a1_modulated.value else None, num_frames, duration)
            decay_np = generate_parameter('decay', ks_burst_decay_center.value, lfo if ks_burst_decay_modulated.value else None, num_frames, duration)
            
            params = {'f0': f0_np, 'a1': a1_np, 'decay': decay_np}
            
            f0_torch = torch.from_numpy(f0_np).float().unsqueeze(0)
            a1_torch = torch.from_numpy(a1_np).float().unsqueeze(0)
            decay_torch = torch.from_numpy(decay_np).float().unsqueeze(0)
            
            f0_upsampled = lin_resample(f0_torch, num_samples)
            a1_upsampled = lin_resample(a1_torch, num_samples)
            decay_upsampled = lin_resample(decay_torch, num_samples)
            
            # Generate burst noise with controlled rate
            x = generate_burst_noise(num_samples, num_frames, trigger_rate, duration)
            
            print(f"⏳ Computing time-domain implementation (burst rate: {trigger_rate:.1f} Hz)...")
            audio_td = karplus_strong(
                x=x,
                f0=f0_upsampled,
                a1=a1_upsampled,
                g=decay_upsampled,
                implementation=Implementation.DIFFABLE_TIME_DOMAIN,
                fs=fs,
                lagrange_order=LAGRANGE_ORDER,
                iir_truncation=IIR_TRUNCATION
            ).squeeze(0).numpy()
            
            print("⏳ Computing frequency-domain implementation...")
            hop_length = N_FFT // 4
            window = torch.hann_window(N_FFT, device=x.device)
            
            X = torch.stft(x, n_fft=N_FFT, hop_length=hop_length, window=window, return_complex=True)
            num_stft_frames = X.shape[-1]
            X = X.permute(0, 2, 1)
            
            f0_stft = lin_resample(f0_torch, num_stft_frames)
            a1_stft = lin_resample(a1_torch, num_stft_frames)
            decay_stft = lin_resample(decay_torch, num_stft_frames)
            
            X_filtered = karplus_strong(
                x=X,
                f0=f0_stft,
                a1=a1_stft,
                g=decay_stft,
                implementation=Implementation.FREQUENCY_SAMPLING,
                fs=fs,
                n_fft=N_FFT
            )
            
            X_filtered = X_filtered.permute(0, 2, 1)
            audio_freq = torch.istft(X_filtered, n_fft=N_FFT, hop_length=hop_length, 
                                    window=window, length=num_samples).squeeze(0).numpy()
            
            param_labels = [
                ('f0', 'f0 (Hz)', ks_burst_f0_modulated.value),
                ('a1', 'a1 (Brightness)', ks_burst_a1_modulated.value),
                ('decay', 'Decay (Sustain)', ks_burst_decay_modulated.value)
            ]
            
            modulated_params = [name for name, _, mod in param_labels if mod]
            mod_str = ', '.join(modulated_params) if modulated_params else 'None'
            title = f"Karplus-Strong Comparison (BURST RATE: {trigger_rate:.1f} Hz)\nLFO: {ks_burst_lfo_rate.value:.1f}Hz @ {ks_burst_lfo_amp.value:.0%} → [{mod_str}]"
            
            plot_comparison(audio_td, audio_freq, params, duration, fs, num_frames, title, param_labels)
            
            stats = compute_stats(audio_td, audio_freq, 'Time-Domain vs Frequency-Domain')
            print("\n" + "="*80)
            print("📊 COMPARISON STATISTICS")
            print("="*80)
            print(f"SNR:              {stats['snr']:>10.2f} dB")
            print(f"RMS Error:        {stats['rms_error']:>10.6f}")
            print(f"Max Abs Diff:     {stats['max_diff']:>10.6f}")
            print(f"Correlation:      {stats['correlation']:>10.8f}")
            match = '✅ MATCH' if stats['snr'] > 60 else '⚠️ DIFFERS'
            print(f"Match:            {match}")
            print("="*80)
            
            print("\n🔊 Audio Playback:")
            print("\nTime-Domain Implementation:")
            display(ipd.Audio(audio_td, rate=fs))
            print("\nFrequency-Domain Implementation:")
            display(ipd.Audio(audio_freq, rate=fs))
            print("\nDifference (Time - Freq):")
            display(ipd.Audio(audio_td - audio_freq, rate=fs))
            
        except Exception as e:
            clear_output(wait=False)
            print("\n" + "="*60)
            print("❌ ERROR OCCURRED:")
            print("="*60)
            print(f"Error type: {type(e).__name__}")
            print(f"Error message: {str(e)}")
            print("\nFull traceback:")
            traceback.print_exc()
            print("="*60)

ks_burst_generate_button.on_click(generate_ks_burst_comparison)

burst_box = widgets.VBox([
    widgets.HTML("<h4>💥 Burst Control</h4>"),
    ks_burst_trigger_rate,
])

lfo_box = widgets.VBox([
    widgets.HTML("<h4>🌊 LFO Settings</h4>"),
    ks_burst_lfo_rate,
    ks_burst_lfo_amp,
])

params_box = widgets.VBox([
    widgets.HTML("<h4>🎛️ Parameters</h4>"),
    ks_burst_duration,
    widgets.HBox([ks_burst_f0_center, ks_burst_f0_modulated]),
    widgets.HBox([ks_burst_a1_center, ks_burst_a1_modulated]),
    widgets.HBox([ks_burst_decay_center, ks_burst_decay_modulated]),
])

controls = widgets.VBox([
    widgets.HTML("<h3>🎹 Karplus-Strong Test (Burst Rate Control)</h3>"),
    widgets.HBox([burst_box, lfo_box]),
    params_box,
    widgets.HTML("<br>"),
    ks_burst_generate_button,
], layout=widgets.Layout(padding='10px', border='2px solid #FF9800'))

display(controls, ks_burst_output)

Output()